# CDP Full Cities Dataset — Passenger Mode Share data, EU Capitals Only 

In [1]:
import requests #makes http calls
import pandas as pd #json responses to df
import time #pauses between paginated requests
import re #regex

## CONFIG

In [2]:
EU_CAPITALS = {
    "Vienna", "Brussels", "Sofia", "Zagreb", "Nicosia", "Prague",
    "Copenhagen", "Tallinn", "Helsinki", "Paris", "Berlin", "Athens",
    "Budapest", "Dublin", "Rome", "Riga", "Vilnius", "Luxembourg",
    "Valletta", "Amsterdam", "Warsaw", "Lisbon", "Bucharest",
    "Bratislava", "Ljubljana", "Madrid", "Stockholm",
}

REGION = "Europe"

## 2024, 2025
These datasets are only accessible on excel format. We will upload the spreadhseet on
air pollution and merge them with the existing df

In [3]:
air_2025 = pd.read_excel("2025_Europe_Cities_Public_Data_Separated_by_Question.xlsx", sheet_name="Q4.5", engine="openpyxl")
air_2024 = pd.read_excel("2024_Full_Cities_Public_Data_Separated_By_Question_CDP_ICLEI_Track.xlsx", sheet_name="Q4.5", engine="openpyxl")
combined = pd.concat([air_2025, air_2024], ignore_index=True)

In [4]:
air_2025

,disclosure_cycle,cdp_requesting_org_number,requesting_organization,cdp_disclosing_org_number,disclosing_organization,disclosing_org_type,cdp_region,discloser_country_or_area,public_status,question_number,...,col15_Freight mode share: Motorcycle / Two wheeler (%),col16_Freight mode share: Light Goods Vehicles (LGV) (%),col17_Freight mode share: Medium Goods vehicles (MGV) (%),col18_Freight mode share: Heavy Goods vehicles (HGV) (%),col19_Freight mode share: Rail (%),col20_Freight mode share: Inland water transport (%),col21_Freight mode share: Other (%),col22_Year freight mode share data applies to,col23_Total freight mode share reported (%),col24_Comment
0,2025 Disclosure Cycle,779465,CDP,1499,City of Barcelona,City,Europe,Spain,Public,Q4.5,...,0.0,75.7,19.04,5.26,0.0,0.0,0.0,2024.0,100.0,Due to lack of data in rail systems and motorb...
1,2025 Disclosure Cycle,779465,CDP,3429,City of Stockholm,City,Europe,Sweden,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data: main mode of trips to work and school by...
2,2025 Disclosure Cycle,779465,CDP,14088,Oslo Municipality,City,Europe,Norway,Public,Q4.5,...,NaN,79.0,NaN,21.00,NaN,NaN,NaN,2024.0,100.0,Modal share report is on daily travel from mon...
3,2025 Disclosure Cycle,779465,CDP,19233,Municipality of Torres Vedras,City,Europe,Portugal,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A informação atual disponível não permite a qu...
4,2025 Disclosure Cycle,779465,CDP,31009,Copenhagen Municipality,City,Europe,Denmark,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Source: Mobilitetsredegørelse for year 2024 (c...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,2025 Disclosure Cycle,779465,CDP,2024876,Town of Lügde,City,Europe,Germany,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
119,2025 Disclosure Cycle,779465,CDP,2044749,Kalix Municipality,City,Europe,Sweden,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,2025 Disclosure Cycle,779465,CDP,2044752,Ystad Municipality,City,Europe,Sweden,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,2025 Disclosure Cycle,779465,CDP,2045345,Municipality of Terézváros,City,Europe,Hungary,Public,Q4.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,The given data is based on the figures from th...


In [5]:
#create column "year"
combined["year"] = combined["disclosure_cycle"].astype(str).str.strip().str.split().str[0]

In [6]:
#Filter by cities (= EU capitals) & region (= Europe)
# No need to filter by Q, since we have uploaded the right spreadsheet

regex_pat = r"\b(?:{})\b".format("|".join(map(re.escape, EU_CAPITALS)))
mask_city = combined["disclosing_organization"].str.contains(regex_pat, case=False, na=False, regex=True)

mask_region = combined["cdp_region"].isin([REGION])

combined_mask = mask_city & mask_region

combined = combined[combined_mask].reset_index(drop=True)

# Cleaning

In [7]:
combined.columns

Index(['disclosure_cycle', 'cdp_requesting_org_number',
       'requesting_organization', 'cdp_disclosing_org_number',
       'disclosing_organization', 'disclosing_org_type', 'cdp_region',
       'discloser_country_or_area', 'public_status', 'question_number',
       'question_text', 'row_order', 'row_name',
       'col1_Passenger mode share data to report',
       'col2_Passenger mode share: Walking (%)',
       'col3_Passenger mode share: Cycling (%)',
       'col4_Passenger mode share: Micromobility (including e-scooters) (%)',
       'col5_Passenger mode share: Buses (including Bus Rapid Transit) (%)',
       'col6_Passenger mode share: Rail/Metro/Tram (%)',
       'col7_Passenger mode share: Ferries/ River boats (%)',
       'col8_Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles) (%)',
       'col9_Passenger mode share: Private motorized transport (%)',
       'col10_Passenger mode share: Informal/paratransit/popular transit systems (%)',
       'col11_Passenger

In [8]:
combined = combined[['disclosing_organization', 'discloser_country_or_area', 'year', 
       'col1_Passenger mode share data to report',
       'col2_Passenger mode share: Walking (%)',
       'col3_Passenger mode share: Cycling (%)',
       'col4_Passenger mode share: Micromobility (including e-scooters) (%)',
       'col5_Passenger mode share: Buses (including Bus Rapid Transit) (%)',
       'col6_Passenger mode share: Rail/Metro/Tram (%)',
       'col7_Passenger mode share: Ferries/ River boats (%)',
       'col8_Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles) (%)',
       'col9_Passenger mode share: Private motorized transport (%)',
       'col10_Passenger mode share: Informal/paratransit/popular transit systems (%)',
       'col11_Passenger mode share: Other (%)',
       'col12_Year passenger mode share data applies to',
       'col13_Total passenger mode share reported (%)',
        'col24_Comment', 
          ]]

In [9]:
combined = combined.rename(columns={
    'disclosing_organization': 'city',
    'discloser_country_or_area': 'country'
})

combined = combined.rename(
    columns=lambda s: re.sub(r'^col\d+_', '', s)   #removes "col<number>_"
)

combined = combined.rename(
    columns=lambda s: re.sub(r' \(%\)$', '', s) #removes (%)
)

In [10]:
combined['city'].unique()

<StringArray>
[             'City of Stockholm',        'Copenhagen Municipality',
                   'City of Rome',         'Municipality of Athens',
           'City-State of Berlin',                 'City of Madrid',
                  'City of Paris',                 'City of Warsaw',
                'City of Tallinn',                 'City of Lisbon',
  'Capital Municipality of Sofia', 'City Municipality of Ljubljana',
 'Zugló District XIV of Budapest',               'City of Brussels',
                 'City of Zagreb']
Length: 15, dtype: str

In [11]:
combined["short_city_name"] = combined["city"].astype(str).str.strip().str.split().str[-1]
combined.loc[combined["city"] == "Copenhagen Municipality", "short_city_name"] = "Copenhagen"

In [12]:
cities2025 = combined[combined["year"] == '2025']['short_city_name'].unique()
cities2025

array(['Stockholm', 'Copenhagen', 'Rome', 'Athens', 'Berlin', 'Madrid',
       'Paris', 'Warsaw', 'Tallinn', 'Lisbon', 'Sofia', 'Ljubljana',
       'Budapest'], dtype=object)

In [13]:
cities2024 = combined[combined["year"] == '2024']['short_city_name'].unique()
cities2024

array(['Sofia', 'Ljubljana', 'Brussels', 'Lisbon', 'Madrid', 'Paris',
       'Rome', 'Stockholm', 'Tallinn', 'Warsaw', 'Zagreb', 'Berlin',
       'Copenhagen', 'Athens'], dtype=object)

In [14]:
missing = [x for x in cities2024 if x not in set(cities2025)]
missing

['Brussels', 'Zagreb']

In [15]:
missing_df = combined[combined["short_city_name"].isin(missing)]
missing_df

,city,country,year,Passenger mode share data to report,Passenger mode share: Walking,Passenger mode share: Cycling,Passenger mode share: Micromobility (including e-scooters),Passenger mode share: Buses (including Bus Rapid Transit),Passenger mode share: Rail/Metro/Tram,Passenger mode share: Ferries/ River boats,Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles),Passenger mode share: Private motorized transport,Passenger mode share: Informal/paratransit/popular transit systems,Passenger mode share: Other,Year passenger mode share data applies to,Total passenger mode share reported,Comment,short_city_name
15,City of Brussels,Belgium,2024,Passenger mode share as share of trips (%),24.0,4.0,NaN,NaN,24.0,0.0,NaN,46.0,NaN,2.0,NaN,100.0,NaN,Brussels
23,City of Zagreb,Croatia,2024,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Zagreb


In [16]:
combined_2025 = combined[combined["year"] == '2025']
combined_2025

,city,country,year,Passenger mode share data to report,Passenger mode share: Walking,Passenger mode share: Cycling,Passenger mode share: Micromobility (including e-scooters),Passenger mode share: Buses (including Bus Rapid Transit),Passenger mode share: Rail/Metro/Tram,Passenger mode share: Ferries/ River boats,Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles),Passenger mode share: Private motorized transport,Passenger mode share: Informal/paratransit/popular transit systems,Passenger mode share: Other,Year passenger mode share data applies to,Total passenger mode share reported,Comment,short_city_name
0,City of Stockholm,Sweden,2025,Passenger mode share as share of trips (%),11.0,20.00,0.0,14.00,35.00,0.0,0.00,17.0,0.0,2.0,2024.0,99.0,Data: main mode of trips to work and school by...,Stockholm
1,Copenhagen Municipality,Denmark,2025,Passenger mode share as share of trips (%),23.0,29.00,0.0,3.00,16.00,0.0,0.40,29.0,0.0,0.0,2024.0,100.4,Source: Mobilitetsredegørelse for year 2024 (c...,Copenhagen
2,City of Rome,Italy,2025,Passenger mode share as share of trips (%),23.5,1.30,7.8,0.00,13.80,0.0,0.00,53.6,0.0,0.0,2024.0,100.0,Data year 2024 Survey of residents of the Muni...,Rome
3,Municipality of Athens,Greece,2025,Passenger mode share as share of vehicle dista...,7.7,0.79,12.8,5.81,11.00,0.0,0.00,61.9,0.0,0.0,2023.0,100.0,Taxis are included in private motorized transp...,Athens
4,City-State of Berlin,Germany,2025,Passenger mode share as share of trips (%),34.0,18.00,0.0,0.00,26.00,0.0,0.00,22.0,0.0,0.0,2024.0,100.0,Rail/Metro/Tram also include taxis,Berlin
5,City of Madrid,Spain,2025,Passenger mode share as share of trips (%),28.8,0.80,0.0,0.00,35.20,0.0,0.00,32.9,0.0,2.3,2018.0,100.0,2018 data according to Madrid 360 - Advance th...,Madrid
6,City of Paris,France,2025,Passenger mode share as share of trips (%),44.0,11.00,0.0,0.00,34.00,0.0,0.00,9.0,0.0,2.0,2023.0,100.0,The latest data on modal shares was produced i...,Paris
7,City of Warsaw,Poland,2025,Passenger mode share as share of vehicle dista...,4.8,1.80,0.1,5.60,24.40,0.0,4.40,58.9,0.0,0.0,2022.0,100.0,Passenger private motorized vehicles includes ...,Warsaw
8,City of Tallinn,Estonia,2025,Passenger mode share as share of trips (%),14.0,4.00,1.0,35.00,4.00,0.0,2.00,40.0,NaN,0.0,2024.0,100.0,NaN,Tallinn
9,City of Lisbon,Portugal,2025,Passenger mode share as share of trips (%),20.7,1.30,NaN,15.40,12.00,0.1,NaN,47.6,0.0,2.9,2021.0,100.0,Data from Census 2021.\nThe year in question c...,Lisbon


In [17]:
combined_recent = pd.concat([combined_2025, missing_df], ignore_index=True)

In [18]:
combined_recent = combined_recent.dropna(subset=['Passenger mode share data to report'])

In [19]:
# replaces all nans in the float columns with 0.
#Columns that aren't floats are left untouched

float_cols = combined_recent.select_dtypes(include='float').columns

combined_recent.loc[:, float_cols] = combined_recent.loc[:, float_cols].fillna(0)

In [20]:
means = combined_recent[float_cols].mean()

mean_df = means.to_frame().T            #T:transpose so columns stay the same, rows = 1

mean_df

,Passenger mode share: Walking,Passenger mode share: Cycling,Passenger mode share: Micromobility (including e-scooters),Passenger mode share: Buses (including Bus Rapid Transit),Passenger mode share: Rail/Metro/Tram,Passenger mode share: Ferries/ River boats,Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles),Passenger mode share: Private motorized transport,Passenger mode share: Informal/paratransit/popular transit systems,Passenger mode share: Other,Year passenger mode share data applies to,Total passenger mode share reported
0,23.157143,7.485,1.707143,9.436429,18.012143,0.007143,0.509286,38.835714,0.0,0.807143,1878.285714,99.957143


In [21]:
mean_df['rail_buses'] = mean_df['Passenger mode share: Buses (including Bus Rapid Transit)'] + mean_df['Passenger mode share: Rail/Metro/Tram']
print(mean_df['rail_buses'])

0    27.448571
Name: rail_buses, dtype: float64


In [22]:
combined_recent['rail_buses'] = combined_recent['Passenger mode share: Buses (including Bus Rapid Transit)'] + combined_recent['Passenger mode share: Rail/Metro/Tram']

In [23]:
combined_recent.sort_values(by='rail_buses')

,city,country,year,Passenger mode share data to report,Passenger mode share: Walking,Passenger mode share: Cycling,Passenger mode share: Micromobility (including e-scooters),Passenger mode share: Buses (including Bus Rapid Transit),Passenger mode share: Rail/Metro/Tram,Passenger mode share: Ferries/ River boats,Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles),Passenger mode share: Private motorized transport,Passenger mode share: Informal/paratransit/popular transit systems,Passenger mode share: Other,Year passenger mode share data applies to,Total passenger mode share reported,Comment,short_city_name,rail_buses
11,City Municipality of Ljubljana,Slovenia,2025,Passenger mode share as share of trips (%),28.8,10.10,1.3,10.50,0.00,0.0,0.00,49.3,0.0,0.0,2024.0,100.0,Ten years after the previous travel behaviour ...,Ljubljana,10.50
2,City of Rome,Italy,2025,Passenger mode share as share of trips (%),23.5,1.30,7.8,0.00,13.80,0.0,0.00,53.6,0.0,0.0,2024.0,100.0,Data year 2024 Survey of residents of the Muni...,Rome,13.80
3,Municipality of Athens,Greece,2025,Passenger mode share as share of vehicle dista...,7.7,0.79,12.8,5.81,11.00,0.0,0.00,61.9,0.0,0.0,2023.0,100.0,Taxis are included in private motorized transp...,Athens,16.81
1,Copenhagen Municipality,Denmark,2025,Passenger mode share as share of trips (%),23.0,29.00,0.0,3.00,16.00,0.0,0.40,29.0,0.0,0.0,2024.0,100.4,Source: Mobilitetsredegørelse for year 2024 (c...,Copenhagen,19.00
12,Zugló District XIV of Budapest,Hungary,2025,Passenger mode share as share of trips (%),34.5,2.00,0.5,7.00,16.67,0.0,0.13,39.2,0.0,0.0,2023.0,100.0,The given data is based on the figures of the ...,Budapest,23.67
13,City of Brussels,Belgium,2024,Passenger mode share as share of trips (%),24.0,4.00,0.0,0.00,24.00,0.0,0.00,46.0,0.0,2.0,0.0,100.0,NaN,Brussels,24.00
4,City-State of Berlin,Germany,2025,Passenger mode share as share of trips (%),34.0,18.00,0.0,0.00,26.00,0.0,0.00,22.0,0.0,0.0,2024.0,100.0,Rail/Metro/Tram also include taxis,Berlin,26.00
9,City of Lisbon,Portugal,2025,Passenger mode share as share of trips (%),20.7,1.30,0.0,15.40,12.00,0.1,0.00,47.6,0.0,2.9,2021.0,100.0,Data from Census 2021.\nThe year in question c...,Lisbon,27.40
7,City of Warsaw,Poland,2025,Passenger mode share as share of vehicle dista...,4.8,1.80,0.1,5.60,24.40,0.0,4.40,58.9,0.0,0.0,2022.0,100.0,Passenger private motorized vehicles includes ...,Warsaw,30.00
6,City of Paris,France,2025,Passenger mode share as share of trips (%),44.0,11.00,0.0,0.00,34.00,0.0,0.00,9.0,0.0,2.0,2023.0,100.0,The latest data on modal shares was produced i...,Paris,34.00


In [24]:
most_populated_EU_capitals = ["Berlin",
"Madrid",
"Warsaw",
"Bucharest",
"Paris",
"Rome",
"Antwerp",
"Prague",
"Dublin",
"Sofia"]

#source: "Public - EDJNet Clean Public Transport in Europe - Database"

In [25]:
#We want to know the average of only the most populous cities

combined_top_ten = combined_recent[combined_recent["short_city_name"].isin(most_populated_EU_capitals)]

means_top = combined_top_ten[float_cols].mean()

mean_combined_top_ten = means_top.to_frame().T            #T:transpose so columns stay the same, rows = 1

mean_combined_top_ten['rail_buses'] = mean_combined_top_ten['Passenger mode share: Buses (including Bus Rapid Transit)'] + mean_combined_top_ten['Passenger mode share: Rail/Metro/Tram']
print(mean_combined_top_ten['rail_buses'])

0    29.15
Name: rail_buses, dtype: float64


In [26]:
mean_combined_top_ten

,Passenger mode share: Walking,Passenger mode share: Cycling,Passenger mode share: Micromobility (including e-scooters),Passenger mode share: Buses (including Bus Rapid Transit),Passenger mode share: Rail/Metro/Tram,Passenger mode share: Ferries/ River boats,Passenger mode share: Taxis or shared vehicles (e.g. hire vehicles),Passenger mode share: Private motorized transport,Passenger mode share: Informal/paratransit/popular transit systems,Passenger mode share: Other,Year passenger mode share data applies to,Total passenger mode share reported,rail_buses
0,26.75,5.6,1.383333,6.9,22.25,0.0,0.766667,35.616667,0.0,0.733333,2022.166667,100.0,29.15
